# 🏛️ VIETNAMESE CULTURAL HERITAGE VQA - LORA V2 TRAINING PIPELINE
**Mô hình:** Vintern-1B-v3.5 (`5CD-AI/Vintern-1B-v3_5`)  
**Cấu hình tối ưu v2:**
- **Thị giác (Vision):** `--max-patches 4` (1,280 visual tokens) bảo toàn chi tiết hoa văn, hoành phi, mái đình.
- **Số vòng học (Epochs):** 2 Epochs (~614 steps) giúp mô hình ghi nhớ và liên kết sâu kiến trúc di sản.
- **Đơn GPU T4 an toàn:** Single GPU (`CUDA_VISIBLE_DEVICES=0`), `batch_size=1`, `grad_accum=16`, triệt tiêu hoàn toàn lỗi DDP & OOM.
- **Đánh giá tự động:** Đầy đủ chỉ số Lexical (F1, ROUGE-L, BLEU-1, BLEU-4) + Giám khảo AI `qwen/qwen3.8-27b` qua Groq API.

In [ ]:
# ==============================================================================
# BƯỚC 1: TỰ ĐỘNG TÌM KIẾM VÀ KIỂM TRA DỮ LIỆU ĐẦU VÀO
# ==============================================================================
from pathlib import Path

# Tự động quét tìm thư mục chứa train.json trong /kaggle/input (Bất kể tên tài khoản nào)
found_train = list(Path("/kaggle/input").rglob("train.json"))
if found_train:
    DATASET_DIR = found_train[0].parent
    print(f"✅ Tự động phát hiện thư mục dữ liệu: {DATASET_DIR}")
else:
    raise FileNotFoundError("❌ Chưa tìm thấy dữ liệu trong /kaggle/input! Hãy kiểm tra xem bạn đã bấm '+ Add Input' để thêm dataset vào Notebook chưa.")

for f in sorted(DATASET_DIR.iterdir()):
    print(f" - {f.name}")

In [ ]:
# ==============================================================================
# BƯỚC 2: CLONE HOẶC CẬP NHẬT MÃ NGUỒN MỚI NHẤT TỪ GITHUB
# ==============================================================================
import os
%cd /kaggle/working
if not os.path.exists("Vietnamese-cultural-heritage-vqa"):
    !git clone -b feat/qlora-training https://github.com/giabaots12/Vietnamese-cultural-heritage-vqa.git
else:
    %cd Vietnamese-cultural-heritage-vqa
    !git reset --hard HEAD
    !git checkout feat/qlora-training
    !git pull origin feat/qlora-training
    %cd /kaggle/working
%cd /kaggle/working/Vietnamese-cultural-heritage-vqa
!git log -2 --oneline

In [ ]:
# ==============================================================================
# BƯỚC 3: CÀI ĐẶT MÔI TRƯỜNG & LOẠI BỎ THƯ VIỆN XUNG ĐỘT (TORCHAO)
# ==============================================================================
!pip uninstall -y torchao
!pip install -q "transformers>=4.48.0,<=4.50.0" accelerate timm einops peft bitsandbytes sentencepiece matplotlib requests
print("✅ Đã thiết lập xong môi trường!")

In [ ]:
# ==============================================================================
# BƯỚC 4: XÁC THỰC GPU TỐC ĐỘ CAO (TESLA T4)
# ==============================================================================
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")

In [ ]:
# ==============================================================================
# BƯỚC 5: KHỞI CHẠY HUẤN LUYỆN LORA V2 (4 PATCHES, 2 EPOCHS, ~6.5H)
# ==============================================================================
from pathlib import Path
if "DATASET_DIR" not in globals():
    DATASET_DIR = list(Path("/kaggle/input").rglob("train.json"))[0].parent

train_path = str(DATASET_DIR / "train.json")
val_path = str(DATASET_DIR / "val.json")
images_path = str(DATASET_DIR / "images")
output_path = "/kaggle/working/lora_finetuned"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python scripts/train_lora.py \
    --model-id "5CD-AI/Vintern-1B-v3_5" \
    --train-data "$train_path" \
    --val-data "$val_path" \
    --images-dir "$images_path" \
    --output-dir "$output_path" \
    --no-4bit \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 2 \
    --lr 1.5e-4 \
    --max-patches 4 \
    --save-steps 200 \
    --eval-steps 200

In [ ]:
# ==============================================================================
# BƯỚC 6: XUẤT BIỂU ĐỒ HỘI TỤ LOSS (TRAINING LOSS & VALIDATION LOSS)
# ==============================================================================
import json
import matplotlib.pyplot as plt
from pathlib import Path

ckpt_dir = Path("/kaggle/working/lora_finetuned/checkpoints")
state_files = list(ckpt_dir.rglob("trainer_state.json"))
if state_files:
    state_files.sort(key=lambda p: p.stat().st_mtime)
    with open(state_files[-1], "r", encoding="utf-8") as f:
        state_data = json.load(f)
    log_history = state_data.get("log_history", [])
    train_steps, train_loss = [], []
    eval_steps, eval_loss = [], []
    for entry in log_history:
        if "loss" in entry and "step" in entry:
            train_steps.append(entry["step"])
            train_loss.append(entry["loss"])
        if "eval_loss" in entry and "step" in entry:
            eval_steps.append(entry["step"])
            eval_loss.append(entry["eval_loss"])
    plt.figure(figsize=(9, 4.5))
    if train_loss:
        plt.plot(train_steps, train_loss, label="Training Loss (4 Patches, 2 Epochs)", color="#1f77b4", linewidth=2)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, "o--", label="Validation Loss", color="#ff7f0e", linewidth=2)
    plt.title("Đường Cong Hội Tụ Loss (Vintern-1B LoRA v2)", fontsize=13, fontweight="bold")
    plt.xlabel("Training Steps")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    chart_path = "/kaggle/working/lora_finetuned/training_curves.png"
    plt.savefig(chart_path, dpi=300)
    plt.show()
    print(f"✅ Đã vẽ và lưu biểu đồ hội tụ: {chart_path}")

In [ ]:
# ==============================================================================
# BƯỚC 7: SUY LUẬN TRÊN TOÀN BỘ 1,060 CÂU KIỂM THỬ (TEST SET)
# ==============================================================================
from pathlib import Path
if "DATASET_DIR" not in globals():
    DATASET_DIR = list(Path("/kaggle/input").rglob("train.json"))[0].parent

test_path = str(DATASET_DIR / "test.json")
images_path = str(DATASET_DIR / "images")
lora_adapter = "/kaggle/working/lora_finetuned/final"
test_predictions = "/kaggle/working/lora_finetuned/predictions_test.json"

!CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline_inference.py \
    --model-id "5CD-AI/Vintern-1B-v3_5" \
    --lora-weights "$lora_adapter" \
    --input "$test_path" \
    --images-dir "$images_path" \
    --output "$test_predictions" \
    --limit 1060

In [ ]:
# ==============================================================================
# BƯỚC 8: ĐÁNH GIÁ CHẤT LƯỢNG TOÀN DIỆN VỚI QWEN 27B LLM-AS-A-JUDGE
# ==============================================================================
# Điền Groq API key của bạn vào đây (hoặc lấy từ Kaggle Secrets/User Secrets)
import os
groq_key = os.environ.get("GROQ_API_KEY", "")
if not groq_key:
    # Bạn có thể điền trực tiếp key vào biến bên dưới trên giao diện Kaggle
    groq_key = ""  # <-- Dán gsk_... của bạn vào đây

!python scripts/evaluate_predictions.py \
    --predictions "/kaggle/working/lora_finetuned/predictions_test.json" \
    --output-dir "/kaggle/working/lora_finetuned" \
    --with-llm-judge \
    --judge-model "qwen/qwen3.8-27b" \
    --api-key "$groq_key"

In [ ]:
# ==============================================================================
# BƯỚC 9: ĐÓNG GÓI TOÀN BỘ KẾT QUẢ VÀ TRỌNG SỐ THÀNH FILE ZIP
# ==============================================================================
!zip -r /kaggle/working/heritage_vqa_finetuned_full_results.zip /kaggle/working/lora_finetuned
print("🎉 ĐÃ HOÀN THÀNH 100%! Tải file: heritage_vqa_finetuned_full_results.zip")